In [1]:
import numpy
import pandas
import evalhyd

## Calculate Brier score and Brier skill score at once

In [2]:
# streamflow observations
obs = numpy.array(
    [[4.7, 4.3, 5.5, 2.7, 4.1]]
)
# streamflow forecasts (3 members)
frc = numpy.array(
    [[5.3, 4.2, 5.7, 2.3, 3.1],
     [4.3, 4.2, 4.7, 4.3, 3.3],
     [5.3, 5.2, 5.7, 2.3, 3.9]]
)
# streamflow thresholds (2)
thr = [4., 5.]

print(obs.shape, frc.shape)

(1, 5) (3, 5)


In [3]:
bs, bss = evalhyd.evalp(obs, frc, ['BS', 'BSS'], thr)

In [4]:
print(
    pandas.DataFrame(
        bs, [f"q > {thr[0]}", f"q > {thr[1]}"],
        ["Brier score"]
    )
)

         Brier score
q > 4.0     0.222222
q > 5.0     0.133333


In [5]:
print(
    pandas.DataFrame(
        bss, [f"q > {thr[0]}", f"q > {thr[1]}"],
        ["Brier skill score"]
    )
)

         Brier skill score
q > 4.0          -0.388889
q > 5.0           0.166667


## Calculate decompositions of Brier score

In [6]:
bs_crd, bs_lbd = evalhyd.evalp(obs, frc, ['BS_CRD', 'BS_LBD'], [4., 5.])

### calibration-refinement decomposition

In [7]:
print(
    pandas.DataFrame(
        bs_crd, [f"q > {thr[0]}", f"q > {thr[1]}"],
        ["reliability", "resolution", "uncertainty"]
    )
)

         reliability  resolution  uncertainty
q > 4.0     0.222222        0.16         0.16
q > 5.0     0.033333        0.06         0.16


#### bs = rel - res + unc?

In [8]:
numpy.allclose(
    numpy.squeeze(bs),
    bs_crd[:, 0] - bs_crd[:, 1] + bs_crd[:, 2]
)

True

### likelihood-base rate decomposition

In [9]:
print(
    pandas.DataFrame(
        bs_lbd, [f"q > {thr[0]}", f"q > {thr[1]}"],
        ["type 2 bias", "discrimination", "sharpness"]
    )
)

         type 2 bias  discrimination  sharpness
q > 4.0     0.072222        0.027778   0.177778
q > 5.0     0.072222        0.027778   0.088889


#### bs = t2b - dis + shr?

In [10]:
numpy.allclose(
    numpy.squeeze(bs),
    bs_lbd[:, 0] - bs_lbd[:, 1] + bs_lbd[:, 2]
)

True